# Livrable 2 - Groupe 1

## Contenu du livrable

Le but est de générer les légendes correspondant aux photos débruitées précédemment. Nous nous appuierons sur les CNN et sur les RNN pour traiter nos photos et générer les légendes. 

## Chargement des bibliothèques

In [22]:
import tensorflow as tf

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

import collections
import random
import re
import numpy as np
import os
import time
import json
from glob import glob
from PIL import Image
import pickle
from tqdm import tqdm

## Chargement des données & définition des constantes

In [23]:
data_path = os.path.abspath('./../data/COCO')
annotation_file = data_path + "/annotations/captions_train2014.json"
IMAGE_DIR = data_path + "/train2014/"

In [55]:
VALIDATION_SPLIT = 0.2
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
SEED = 42
BUFFER_SIZE = 1000
embedding_dimension = 256
UNITS = 512
TOP_K = 5000 # Nombre de mots à conserver dans le vocabulaire
FEATURES_SHAPE = 2048
ATTENTION_FEATURES_SHAPE = 64
EPOCHS = 20
CHECKPOINT_PATH = "./checkpoints/train"

### Pré-traitement des annotations

In [25]:
with open(annotation_file, 'r') as f:
    annotations = json.load(f)

# Grouper toutes les annotations ayant le meme identifiant.
image_path_to_caption = collections.defaultdict(list)
for val in annotations['annotations']:
    # marquer le debut et la fin de chaque annotation
    caption = '<start> ' + val['caption'] + ' <end>'
    # L'identifiant d'une image fait partie de son chemin d'accès
    image_path = IMAGE_DIR + 'COCO_train2014_' + '%012d.jpg' % (val['image_id'])
    # Rajout du caption associé à image_path
    image_path_to_caption[image_path].append(caption)

image_paths = list(image_path_to_caption.keys())

captions = [] # On va stocker tous les captions associés à chaque image
img_name_vector = [] # On va stocker le nom de chaque image pour chaque caption associé
for img_path in image_paths:
    # On rajoute tous les captions associés à l'image
    captions.extend(image_path_to_caption[img_path])
    img_name_vector.extend([img_path] * len(image_path_to_caption[img_path])) # On rajoute le meme nom d'image pour chaque caption associé

### Pré-traitement des images

In [26]:
def load_image(image_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.keras.applications.inception_v3.preprocess_input(image) # TODO : check if this is the right preprocessing
    return image, image_path

In [27]:
img_names = sorted(set(img_name_vector)) # On va stocker le nom de chaque image pour chaque caption associé
image_dataset = tf.data.Dataset.from_tensor_slices(img_names) # On va créer un dataset à partir de tous les noms d'images
image_dataset = image_dataset.map(
    load_image, 
    num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE) # On va charger les images à partir de leurs noms

## Chargement du modèle

In [28]:
image_model = tf.keras.applications.InceptionV3(include_top=False,
                                                weights='imagenet')
new_input = image_model.input
hidden_layer = image_model.layers[-1].output
image_features_extract_model = tf.keras.Model(inputs=new_input, 
                                              outputs=hidden_layer)


### Utilisation de InceptionV3 pour pré-traiter les images

In [29]:
for image, path in tqdm(image_dataset):
    batch_features = image_features_extract_model(image) # On va extraire les features de chaque image
    batch_features = tf.reshape(batch_features, 
                                (batch_features.shape[0], 
                                 -1, 
                                 batch_features.shape[3]))
    for bf, p in zip(batch_features, path):
        np.save(p.numpy().decode('utf-8'), bf.numpy())

100%|██████████| 2587/2587 [49:54<00:00,  1.16s/it]


### Tokenisation des légendes

In [32]:
def calculate_max_caption_length(caption):
    return max(len(c.split()) for c in caption) # On va calculer la longueur maximale de chaque caption

def tokenize(caption):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(
        num_words=TOP_K,
        oov_token='<unk>',
        filters='!"#$%&()*+.,-/:;=?@[\\]^_`{|}~ '
    )
    tokenizer.fit_on_texts(caption) # On va créer un tokenizer à partir de tous les captions
    tokenizer.word_index['<pad>'] = 0 # On rajoute un token pour le padding
    tokenizer.index_word[0] = '<pad>' # On rajoute un token pour le padding

    train_sequences = tokenizer.texts_to_sequences(caption) # On va transformer tous les captions en séquences de tokens
    caption_vector = tf.keras.preprocessing.sequence.pad_sequences(
        train_sequences, 
        padding='post', 
        maxlen=calculate_max_caption_length(caption)) # On va rajouter du padding à chaque séquence de tokens
    
    max_length = calculate_max_caption_length(caption) # On va calculer la longueur maximale de chaque caption

    return caption_vector, max_length, tokenizer # On va retourner le vecteur de captions, la longueur maximale et le tokenizer

captions_vector, max_length, tokenizer = tokenize(captions) # On va créer un tokenizer à partir de tous les captions

## Création du jeu d'entraînement et de test

Nous séparons le jeu de données en deux parties : 
- un jeu d'entraînement (80%)
- un jeu de validation (20% qui correspond au `VALIDATION_SPLIT`)

In [ ]:
image_to_caption_vector = collections.defaultdict(list) # On va créer un dictionnaire à partir de tous les captions
for image, caption in zip(img_name_vector, captions_vector):
    image_to_caption_vector[image].append(caption) # On va rajouter le caption associé à l'image

# Fractionnement du dataset
image_keys = list(image_to_caption_vector.keys()) # On va créer une liste à partir de tous les noms d'images
random.seed(SEED) # On va définir la graine pour la reproductibilité
random.shuffle(image_keys) # On va mélanger les noms d'images
train_size = int(len(image_keys) * (1 - VALIDATION_SPLIT)) # On va calculer la taille du dataset d'entrainement
train_image_keys, validation_image_keys = image_keys[:train_size], image_keys[train_size:] 


def extract_image_and_caption(image_keys, image_to_caption_vector): 
    captions = []
    images_names = []

    for image in image_keys:
        caption = image_to_caption_vector[image] # On va récupérer tous les captions associés à l'image
        images_names.extend([image] * len(caption)) # On va rajouter le nom de l'image pour chaque caption associé
        captions.extend(caption) # On va rajouter tous les captions associés à l'image
    
    return captions, images_names # On va retourner le vecteur de captions et le nom de chaque image pour chaque caption associé


train_captions, train_image_names = extract_image_and_caption(train_image_keys, image_to_caption_vector) # Création du dataset d'entrainement

validation_captions, validation_image_names = extract_image_and_caption(validation_image_keys, image_to_caption_vector) # Création du dataset de validation



In [35]:
len(train_captions), len(train_image_names), len(validation_captions), len(validation_image_names) # On va afficher la taille du dataset d'entrainement et de validation

(331287, 331287, 82826, 82826)

In [72]:
# Utilisation de tf.data.Dataset pour créer le dataset d'entrainement

def map_func(image_name, caption):
    # On va charger l'image à partir de son nom
    image_tensor = np.load(image_name.decode('utf-8') + '.npy')
    return image_tensor, caption # On va retourner l'image et le caption associé

def create_dataset(images_names, captions):
    dataset = tf.data.Dataset.from_tensor_slices((images_names, captions))
    dataset = dataset.map(
        lambda image, caption: tf.numpy_function(
            map_func,
            [image, caption],
            [tf.float32, tf.int32]
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    
    # Explicitly set the shapes of the tensors
    dataset = dataset.map(
        lambda image_tensor, caption: (
            tf.ensure_shape(image_tensor, [ATTENTION_FEATURES_SHAPE, FEATURES_SHAPE]),
            tf.ensure_shape(caption, [None])
        )
    )
    
    dataset = dataset.map(
        lambda image_tensor, caption: (
            {
                "encoder_input": image_tensor,
                "decoder_input": tf.concat(
                    [tf.expand_dims([tokenizer.word_index['<start>']], 0), tf.expand_dims(caption[:-1], 0)], axis=1
                ),
                "hidden_state_input": tf.zeros((UNITS,))
            },
            caption[1:]  # Target output (shifted captions)
        )
    )
    dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset


dataset = create_dataset(train_image_names, train_captions) # Création du dataset d'entrainement
validation_dataset = create_dataset(validation_image_names, validation_captions) # Création du dataset de validation

## Création de la partie Encodeuse avec des CNN

In [73]:
class CNN_Encoder(tf.keras.Model):
    def __init__(self, embedding_dimension):
        super(CNN_Encoder, self).__init__()
        self.fc = tf.keras.layers.Dense(embedding_dimension) # self.fc est une couche dense qui va transformer les features de l'image en un vecteur de dimension embedding_dimension

    def call(self, x):
        x = self.fc(x)
        x = tf.nn.relu(x)
        return x

## Création du mécanisme d'attention

In [74]:
class BahdanauAttention(tf.keras.Model):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)
    
    def call(self, features, hidden):
        hidden_with_time_axis = tf.expand_dims(hidden, 1) # Updating tensor shape from (batch_size, units) to (batch_size, 1, units)
        attention_hidden_layer = tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis)) # On va calculer la couche

        # Calcul d'un score non normalisé pour chaque caractéristique de l'image
        score = self.V(attention_hidden_layer) 
        attention_weights = tf.nn.softmax(score, axis=1) # On va normaliser les scores pour chaque caractéristique de l'image
        
        context_vector = attention_weights * features # On va multiplier les scores normalisés par les caractéristiques de l'image
        context_vector = tf.reduce_sum(context_vector, axis=1) # On va sommer les caractéristiques de l'image pour obtenir le vecteur de contexte
        
        return context_vector, attention_weights

## Création du décodeur RNN

In [75]:
class RNN_Decoder(tf.keras.Model):
    def __init__(self, embedding_dimension, units, vocabulary_size): 
        super(RNN_Decoder, self).__init__()
        self.units = units
        self.embedding = tf.keras.layers.Embedding(vocabulary_size, 
                                                   embedding_dimension)
        self.gru = tf.keras.layers.GRU(self.units,
                                        return_sequences=True,
                                        return_state=True,
                                        recurrent_initializer='glorot_uniform')
        self.fc1 = tf.keras.layers.Dense(self.units) # self.fc1 est une couche dense qui va transformer le vecteur de contexte en un vecteur de dimension units
        self.fc2 = tf.keras.layers.Dense(vocabulary_size) # self.fc2 est une couche dense qui va transformer le vecteur de contexte en un vecteur de dimension vocabulary_size

        self.attention = BahdanauAttention(self.units) # self.attention est une couche d'attention qui va calculer le vecteur de contexte à partir des caractéristiques de l'image et de l'état caché du RNN

    def call(self, x, features, hidden):
        context_vector, attention_weights = self.attention(features, hidden)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)

        output, state = self.gru(x)

        y = self.fc1(output)
        y = tf.reshape(y, (-1, y.shape[2]))
        y = self.fc2(y)

        return y, state, attention_weights # On va retourner le vecteur de sortie, l'état caché et les poids d'attention
    
    def reset_states(self, batch_size=None):
        return tf.zeros((batch_size, self.units)) # On va réinitialiser l'état caché du RNN

## Combiner l'encodeur et le décodeur

In [76]:
encoder = CNN_Encoder(embedding_dimension) # On va créer un encodeur à partir de la classe CNN_Encoder
decoder = RNN_Decoder(embedding_dimension, UNITS, TOP_K) # On va créer un décodeur à partir de la classe RNN_Decoder

In [77]:
optimizer = tf.keras.optimizers.Adam() # On va créer un optimiseur à partir de la classe Adam

def loss_function(real, pred):
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, 
    reduction='none') # On va créer une fonction de perte à partir de la classe SparseCategoricalCrossentropy
    
    mask = tf.math.logical_not(tf.math.equal(real, 0)) # On va créer un masque pour ignorer les tokens de padding
    loss_ = loss_object(real, pred) # On va calculer la perte
    mask = tf.cast(mask, dtype=loss_.dtype) # On va convertir le masque en type de la perte
    loss_ *= mask # On va multiplier la perte par le masque

    return tf.reduce_mean(loss_) # On va retourner la moyenne de la perte

In [78]:

checkpoint = tf.train.Checkpoint(encoder=encoder,
                                    decoder=decoder,
                                    optimizer=optimizer) # On va créer un point de contrôle à partir de l'encodeur, du décodeur et de l'optimiseur

checkpoint_manager = tf.train.CheckpointManager(checkpoint,
                                                CHECKPOINT_PATH,
                                                max_to_keep=5) # On va créer un gestionnaire de point de contrôle à partir du point de contrôle et du chemin d'accès au point de contrôle

In [79]:
start_epoch = 0
if checkpoint_manager.latest_checkpoint:
    start_epoch = int(checkpoint_manager.latest_checkpoint.split('-')[-1]) # On va récupérer le dernier point de contrôle
    checkpoint.restore(checkpoint_manager.latest_checkpoint) # On va restaurer le dernier point de contrôle
    print("Restored from {}".format(checkpoint_manager.latest_checkpoint)) # On va afficher le dernier point de contrôle restauré

## Entrainement du modèle

In [80]:
loss_plot = []

@tf.function
def train_step(img_tensor, target):
    # img_tensor : image
    # target : caption
    
    loss = tf.constant(0.0, dtype=tf.float32)
    hidden = decoder.reset_states(batch_size=target.shape[0]) # Initialisation de l'état caché pour chaque bactch
    decoder_input = tf.expand_dims([tokenizer.word_index['<start>']] * target.shape[0], 1) # On va créer un vecteur d'entrée pour le décodeur à partir du token <start>

    with tf.GradientTape() as tape:
        features = encoder(img_tensor)

        for i in range(1, target.shape[1]):
            # Passer l'image et le vecteur d'état caché au décodeur
            predictions, hidden, _ = decoder(decoder_input, features, hidden)
            loss += loss_function(target[:, i], predictions)

            decoder_input = tf.expand_dims(target[:, i], 1)
    
    total_loss = (loss / int(target.shape[1])) # On va calculer la perte totale 

    trainable_variables = encoder.trainable_variables + decoder.trainable_variables # On va récupérer les variables entraînables de l'encodeur et du décodeur

    gradients = tape.gradient(total_loss, trainable_variables) # On va calculer les gradients de la perte totale par rapport aux variables entraînables

    optimizer.apply_gradients(zip(gradients, trainable_variables)) # On va appliquer les gradients à l'optimiseur

    return loss, total_loss # On va retourner la perte et la perte totale


In [ ]:
# Define the input shapes for the encoder and decoder
encoder_input = tf.keras.Input(shape=(ATTENTION_FEATURES_SHAPE, FEATURES_SHAPE), name="encoder_input")
decoder_input = tf.keras.Input(shape=(None,), name="decoder_input")
hidden_state_input = tf.keras.Input(shape=(UNITS,), name="hidden_state_input")

# Call the encoder and decoder with the defined inputs
encoder_output = encoder(encoder_input)
decoder_output, _, _ = decoder(decoder_input, encoder_output, hidden_state_input)

# Create the model
model = tf.keras.Model(
    inputs=encoder_input,
    outputs=decoder_output) # On va créer un modèle à partir de l'encodeur et du décodeur

model.compile(optimizer=optimizer, 
              loss=loss_function, 
              metrics=['accuracy']) # On va compiler le modèle avec l'optimiseur et la fonction de perte

history = model.fit(dataset, 
                    epochs=EPOCHS, 
                    steps_per_epoch=len(train_captions)//BATCH_SIZE, 
                    validation_data=validation_dataset, 
                    validation_steps=len(validation_captions)//BATCH_SIZE) # On va entraîner le modèle avec le dataset d'entrainement et de validation

NotImplementedError: Exception encountered when calling Lambda.call().

[1mWe could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.[0m

Arguments received by Lambda.call():
  • args=(['<KerasTensor shape=(None,), dtype=float32, sparse=False, ragged=False, name=keras_tensor_1002>', '<KerasTensor shape=(None, 64, 256), dtype=float32, sparse=False, ragged=False, name=keras_tensor_999>', '<KerasTensor shape=(None, 512), dtype=float32, sparse=False, ragged=False, name=hidden_state_input>'],)
  • kwargs={'mask': ['None', 'None', 'None']}